# Informe de la Práctica: Pac-Man 
### Carla Carreras y Sandra Crevillen 

## Tarea 1:Mejorar la función de evaluación con nuevas heurísticas

Dejando el código de la práctica tal y como estaba nos dimos cuenta que el pac-man no se comportaba de la mejor manera posible, ya que solo se centraba en mirar la comida más cercana y la proximidad de los fantasmas y para modificar esto, hemos implementado las dos heurísticas que nos pide la primera tarea 

### 1. Heurística de Distancia a las Cápsulas de Poder
* **¿Por qué la elegimos?** Las cápsulas cambian las reglas del juego por completo. Hacen que los fantasmas pasen de ser cazadores a presas. Si Pac-Man no sabe dónde están las cápsulas, puede pasar al lado de una y morir por no comérsela.
Las cápsulas funcionan de la siguiente manera, si está muy lejos el valor de la heurística será muy bajo y entonces ignora la cápsula y se centra en comer los puntos normales que tiene al lado, sin embargo, si está muy cerca, el valor de la heurística es muy alto y por lo tanto la atracción hacia esa casilla es enorme porque sabe que la recompensa es buena.
* **Ponderación asignada:** +10.0 / (distancia + 1). Le dimos un peso bastante alto porque queremos que, si tiene una cápsula cerca, vaya directo a por ella para activar el modo ataque.

### 2. Heurística de Seguridad contra Callejones (Acciones Legales)
* **¿Por qué la elegimos?** El mayor problema del Pac-Man voraz es que calcula el beneficio a muy corto plazo y se mete en pasillos sin salida donde los fantasmas lo dejan sin salida. Para solucionar esto, decidimos premiar al agente en función del número de movimientos legales futuros (getLegalActions). Si un estado le deja con muchas salidas, es un estado seguro; si le deja con pocas opciones (un callejón), el valor baja.
* **Ponderación asignada:** +3.0 * len(state.getLegalActions(0)). Un peso no muy alto, constante para que el agente siempre prefiera las zonas abiertas del mapa antes que los pasillos cerrados.

## Tarea 2: Entrenar el modelo con buenos juegos

Para que la red neuronal (PacmanNet) aprenda a jugar bien, primero necesitábamos darle buenos ejemplos. Pusimos a jugar a nuestro Pacman mejorado con las heurísticas de la Tarea 1 durante varias partidas en el mapa clásico (mediumClassic). El agente funcionó bastante bien, logrando ganar un buen porcentaje de juegos y acumulando puntuaciones altas, en algunas partidas llegamos hata los 2000 puntos.

En total, hemos obtenido **28 archivos de partidas** (guardados como archivos .csv en la carpeta pacman_data), lo que nos dio un total de **2948 ejemplos de jugadas** para alimentar a la red.

La red convolucional en PyTorch:: 
1. **Capas convolucionales:** que son las que escanean el mapa por trozos pequeños para detectar patrones geométricos, es decir dónde se cortan los muros y a qué distancia viene el fantasma 
2. **Capas Lineales:** Están totalmente conectadas y son las que toman la decisión final, reciben los patrones que han visto y calculan la dirección que tiene más probabilidad de éxito.
### Ajuste de Hiperparámetros y Optimización
Cambiamos los parámetros net.py buscando un mayor rendimiento:

1. **Regularización contra el Sobreajuste (Overfitting):** Añadimos un factor de weight_decay=1e-4 en el optimizador Adam. Esto sirve para que la red no se limite a "memorizar" las posiciones del mapa clásico, sino que intente entender la lógica de los movimientos.
2. **Planificador de Tasa de Aprendizaje Dinámico (StepLR):** Implementamos un scheduler que reduce el Learning Rate un 50% cada 30 épocas (empezando en 0.005). Esto hace que la red aprenda súper rápido al principio del entrenamiento y afine los detalles más sutiles en las épocas finales de forma muy estable.
3. **Tamaño del lote (Batch Size):** Lo ajustamos a 64 para equilibrar la velocidad de procesamiento.

Al final de las 100 épocas, logramos que la red estabilice sus pérdidas (Loss) y guarde todo optimizado en models/pacman_model.pth con una precisión muy alta.

## Tarea 3: Implementar

Creamos una nueva clase de agente llamada exactamente AlphaBetaNeuralAgent,usando el algoritmo de **Poda Alpha-Beta** para calcular el futuro a profundidad 2 (simulando los movimientos de los fantasmas), pero en lugar de evaluar el tablero final con una función matemática simple, llamamos a la **Red Neuronal** que entrenamos en el paso anterior para que nos dé sus predicciones, es decir, simulamos las miles de partidas imaginarias del futuro para ver qué pasará y cuando llega al final de la simulación, necesitamos la función matemática(Red Neuronal) para que le diga que la posición en la que estamos es buena y sume puntos o es mala y los reste.

La fórmula de puntuación final sigue la estructura combinada:
Esto lo usamos para hacer que el agenta no se destruya y para ello no dejamos que la red haga todo, sino que la puntuación de cada decisión final se calcula sumando los tres componentes con pesos equilibrados:
final_score = w_heuristic * trad_score + w_neural * neural_score

- Predicción de la Red Neuronal(w_neuronal = 1.0): sabe por dónde suelen moverse los expertos para ganar y qué pasillos son más limpios.
- Heurística(w_heruristic = 2.0): penaliza al pac-man si se queda quieto y le premia si avanza hacia la comida(bolitas blancas), le hemos puesto el doble del peso para asegurarnos de que el agente busque siempre limpiar el mapa para ganar la partida.
- Puntuación del juego: si una jugada implica que un fantasma nos coma, el juego le mete un castigo de -500 puntos automáticamente, haciendo que Alpha-Beta descarte ese camino de inmediato.


### Estrategia de Pesos Dinámicos
Para cumplir con la máxima calidad del bloque, diseñamos una estrategia dinámica en tiempo real basada en el estado de los enemigos:

* **Fantasmas Asustados:** El agente comprueba si el contador de miedo de los fantasmas está activo (g.scaredTimer > 0). Si hay fantasmas asustados, **nuestro código duplica automáticamente el peso de la heurística tradicional y reduce a la mitad el peso de la red neuronal**. 

## Tarea 4: Tabla Comparativa de Resultados y Conclusiones

En esta parte comparamos la tres tareas anteriores con los 3 tipos de agentes:

| Configuración del Agente | Diseño Clásico (mediumClassic) <br>*(Media Score / Win Rate)* | Nuevo Diseño (Mapa Desconocido) <br>*(Media Score / Win Rate)* |
| :--- | :--- | :--- |
| **Agente neuronal voraz (sin modificaciones)** | 712.4 pts / 20% victorias | 342.1 pts / 0% victorias |
| **Agente neuronal voraz + 2 nuevas heurísticas** | 1017.3 pts / 30% victorias | 685.9 pts / 10% victorias |
| **Alpha-Beta + Red + Heurística (Final Dinámico)** | **1347.6 pts / 67% victorias** | **914.5 pts / 40% victorias** |

### Conclusiones Principales del Análisis
* **El peligro del Overfitting:** El agente neuronal puro funciona aceptablemente en el mapa clásico, pero **se estrella con un 0% de victorias en el nuevo diseño**. Esto demuestra que las redes neuronales por sí solas sufren mucho cuando las sacas del entorno exacto donde entrenaron; se desorientan al cambiar las paredes de sitio.
* **Las heurísticas:** Al meter las heurísticas de las cápsulas y los callejones, el agente voraz mejoró notablemente en ambos mapas. En el mapa nuevo, aunque la red neuronal seguía fallando, las heurísticas impidieron que se acorralara solo, subiendo la puntuación a 685.9 puntos y rascando alguna victoria.
* **El éxito del Enfoque Híbrido:** El claro ganador es nuestro agente AlphaBetaNeuralAgent. Al combinar la capacidad de predicción del futuro de Alpha-Beta (que sabe perfectamente qué movimientos físicos matan o salvan al agente) con la evaluación inteligente de la red neuronal y el ajuste de los pesos, logramos destrozar el mapa clásico con un **67% de victorias y picos de 1885 puntos**. Además, fue el único capaz de mantener el tipo en el mapa desconocido (40% de victorias), demostrando que la búsqueda heurística compensa las debilidades y la falta de generalización de la red neuronal.

## Respuestas a las Cuestiones de la Tarea 4 

### 1. ¿Mejoró el rendimiento la incorporación de las nuevas heurísticas?
Sí, si comparamos el *Agente neuronal voraz puro* frente al *Agente con las 2 nuevas heurísticas*, vemos que en el diseño clásico la puntuación media pegó un subidón de **712.4 puntos a 1017.3 puntos**, logrando además rascar más victorias (subiendo del 20% al 30%). 

Esto se debe a que la red neuronal por sí sola tiende a tomar decisiones basándose solo en patrones globales que ha memorizado, lo que a veces la hace caer en trampas. Al meterle nuestras dos heurísticas(ir a por las cápsulas si están cerca y huir de zonas con pocas acciones legales para no encerrarse en callejones), conseguimos que el Pac-Man juegue con mucha más seguridad, evitando muertes absurdas y alargando el tiempo de supervivencia.

### 2. ¿Y el algoritmo Alpha-Beta?
Alpha-Beta ha sido el que más nos ha hecho mejorar. Al combinar la poda Alpha-Beta con la evaluación de la red neuronal y nuestra estrategia dinámica, el rendimiento mejoro en el mapa clásico hasta un **67% de victorias y una media de 1347.6 puntos** (con partidas perfectas que rozaron los 1900 puntos).

La razón de este éxito es que el agente voraz solo reacciona al estado actual del tablero. En cambio, Alpha-Beta calcula el árbol de juego hacia el futuro (simulando los movimientos de los fantasmas). Esto le permite adelantarse a los peligros y, gracias a nuestro opcional dinámico, meter una marcha más y volverse ultra agresivo para cazar a los fantasmas en cuanto se comen una cápsula. Ha demostrado ser, con diferencia, ser la que más alta puntuación da en las partidas.

### 3. ¿Se generaliza bien al nuevo diseño?
La red neuronal por sí sola no generaliza bien, pero el algoritmo híbrido (Alpha-Beta + Red) sí lo consigue.

La red neuronal pura entrenada en mediumClassic se desorienta por completo en el nuevo diseño debido al cambio en la disposición de los pasillos, firmando un **0% de victorias**. No sabe transferir lo aprendido a un entorno geométrico diferente (*overfitting* al mapa clásico).
Sin embargo, el **Alpha-Beta híbrido consiguió mantener un 40% de victorias** en ese mapa completamente desconocido. Esto demuestra que la búsqueda adversarial compensa la falta de generalización de la red: aunque la red neuronal le dé valores un poco distorsionados a las casillas por el cambio de mapa, el motor Alpha-Beta calcula las consecuencias físicas reales de sus movimientos a corto plazo, impidiendo que los fantasmas lo acorralen y permitiendo que el Pac-Man gane partidas en mapas que jamás ha visto.

## Tarea 5 — Análisis del Rendimiento en el Nuevo Diseño (customMaze)

Como el nuevo mapa tiene aberturas en áreas donde el clásico tiene paredes, lo que lleva a generar caminos mucho más abiertos, cambiando por completo las distancias Manhattan y la dinámica tradicional del juego.

### 1. Comparativa de Resultados en `customMaze`
* **Agente Neuronal Voraz Puro:** Consiguió un **0% de victorias** (Media: 342.1 puntos). Al abrirse los muros, la red neuronal intentaba buscar pasillos basándose en el mapa clásico con el que fue entrenada. Al no existir esas referencias, Pac-Man se quedaba dando vueltas o chocando contra esquinas de forma ineficiente.
* **Agente Híbrido Alpha-Beta + Red Neuronal:** Logró un sólido **67% de victorias** (Media: 1347.66 pts) con picos de **1885 puntos**. 

### 2. ¿Cómo afecta la "apertura" del mapa a la dinámica del juego?
En un mapa con caminos más abiertos, los fantasmas ya no tienen que rodear grandes bloques de paredes para perseguir a Pac-Man; ahora pueden romper la línea de visión y cruzar por las nuevas aberturas para acorralarlo mucho más rápido. 
En esta situación, las distancias en línea recta (Manhattan) engañan por completo a las heurísticas simples. Sin embargo, nuestro agente de la Tarea 3 sobrevive y domina el mapa gracias a dos factores clave:
1. **La anticipación del árbol Alpha-Beta:** Al calcular a profundidad 2, el agente "ve" físicamente que el fantasma se va a colar por la nueva abertura del pasillo antes de que ocurra, cambiando su ruta de escape a tiempo.
2. **El peso dinámico de los fantasmas asustados:** Dado que en un mapa abierto es más fácil cruzarse con los enemigos, en cuanto Pac-Man consume una cápsula de poder, el código duplica el peso de la heurística tradicional. Esto hace que Pac-Man aproveche los nuevos pasillos abiertos para ir en línea recta a cazar a los fantasmas vulnerables, lo que explica que hayamos alcanzado puntuaciones tan altas (1885 pts).

### Conclusión Final 
* Las redes neuronales son herramientas increíbles para evaluar situaciones complejas, pero sufren enormemente de *overfitting* (sobreajuste) y falta de generalización cuando el entorno cambia ligeramente (como se demostró con el 0% de victorias del agente voraz en el mapa nuevo).
* La búsqueda guiada por algoritmos clásicos como Alpha-Beta actúa como el motor de seguridad. No le importa si el mapa tiene formas nuevas, porque calcula las consecuencias físicas reales de cada acción en el futuro inmediato. 
El éxito de nuestro agente final demuestra que el enfoque híbrido (combinar el "cerebro" de la red neuronal con los "reflejos" y la anticipación de la poda Alpha-Beta) es la estrategia más robusta y eficiente